In [2]:
from datetime import datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.options.display.float_format = '{:.7g}'.format # 百万の表示まで可能
import matplotlib.pyplot as plt
import seaborn as sns
import re
from mycolor import *
print("code loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

code loading  2019/11/05 00:11:06


In [3]:
# import importlib
# importlib.reload(mr)
import myrace as mr
print("myrace loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

myrace loading  2019/11/05 00:11:09


In [4]:
import urllib.request
from bs4 import BeautifulSoup
""" スクレイピング用の関数 """
def use_urllib(url):
    import urllib.request
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:70.0) Gecko/20100101 Firefox/70.0"}
    req = urllib.request.Request(url, None, headers) # None->timeout=*
    try:
        res = urllib.request.urlopen(req)
    except urllib.error.URLError as e:
        if hasattr(e, 'reason'):
            print('We failed to reach a server.')
            print('Reason: ', e.reason)
        elif hasattr(e, 'code'): # HTTPError
            print('The server couldn\'t fulfill the request.')
            print('Error code: ', e.code)
    else:
        html = res.read()
        return html

def use_selenium(url):
    import warnings
    warnings.filterwarnings('ignore')
    # -> warn('Selenium support for PhantomJS has been deprecated, please use headless '
    from selenium import webdriver
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    # from selenium.webdriver.common.by import By
    from selenium.common.exceptions import TimeoutException
    
    driver = webdriver.PhantomJS()
    driver.get(url)
    for _ in range(3):
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_all_elements_located)
            # WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, 'race_main')))
        except TimeoutException as te:
            print("Web driver: ", te)
        else:
            break
    else:
            driver.close()
            sys.exit()
    html = driver.page_source
    driver.close()
    return html
print("Scraping Function loaded ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

Scraping Function loaded  2019/11/05 00:11:16


In [81]:
import itertools as it

class Betsimulation():
    
    def __init__(self, rc):
        self.rc = rc[2:] # yahoo用
        self._win_soup = None
        self._qui_soup = None
        self._exa_soup = None
        self._wide_soup = None
        self.horse_dic = {}
        self.win_df = pd.DataFrame([])
        self.win_sr = pd.Series([])
        self.show1_sr = pd.Series([])
        self.show2_sr = pd.Series([])
        self.qui_sr = pd.Series([])
        self.exa_sr = pd.Series([])
        self.wide_sr = pd.Series([])
        
    def _set_win_soup(self):
        """ 基本情報の取得 """
        url = "https://keiba.yahoo.co.jp/odds/tfw/" + self.rc + "/"
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        self._win_soup = soup
        return soup
    
    def _set_quinella_soup(self):
        url = "https://keiba.yahoo.co.jp/odds/ur/" + self.rc + "/"
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        self._qui_soup = soup
        return soup
    
    def _set_exacta_soup(self):
        url = "https://keiba.yahoo.co.jp/odds/ut/" + self.rc + "/"
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        self._exa_soup = soup
        return soup
    
    def _set_wide_soup(self):
        url = "https://keiba.yahoo.co.jp/odds/wide/" + self.rc + "/"
        html = use_urllib(url)
        soup = BeautifulSoup(html,"lxml")
        self._wide_soup = soup
        return soup
    
    def _set_horse_dic(self):
        if(self._win_soup==None): soup = self._set_win_soup()
        else: soup = self._win_soup
        horse_tags =[]
        for table in soup.find_all("table", class_="dataLs oddTkwLs"):
            horse_tags += table.find_all("td", class_="txL")
        horses = [tag.a.string for tag in horse_tags]
        horse_dic ={}
        for i, horse in enumerate(horses):
            horse_dic[str(i+1)] = horse
        self.horse_dic = horse_dic
        return horse_dic
    
    def racename(self):
        if(self._win_soup==None): soup = self._set_win_soup()
        else: soup = self._win_soup
        racename = soup.find('h1').string.strip() + \
        re.sub("[a-zA-Z]+|[<>/=\|\"|]","",str(soup.find('p', id="raceTitDay"))).strip()
        return racename
    
    def winranking(self):
        if(self.win_df.empty): win_df = self.win_dataframe()
        else: win_df = self.win_df
        print(self.racename())
        display(win_df)
    
    def win_dataframe(self): # win_dataframeには複勝も含まれている
        if(self._win_soup==None): soup = self._set_win_soup()
        else: soup = self._win_soup
        df = pd.io.html.read_html(soup.prettify())
        if(len(df) > 3):
            if("馬名" in df[3].columns): # 単勝ではなく枠連の可能性もあるため
                concat_df = pd.concat([df[2], df[3]])
            else: concat_df = df[2]
        else: concat_df =  df[2]
        drop_df = concat_df.drop(columns=["複勝.1"])
        drop_df.columns = ['Bk', 'Hs', 'Horse', 'Win', 'Show1', 'Show2']
        sort_df = drop_df.sort_values('Win')
        win_df = sort_df.reset_index()
        self.win_df = win_df
        return win_df
    
    def win_series(self):
        if(self.win_df.empty): win_df = self.win_dataframe()
        else: win_df = self.win_df
        idx = list(map(lambda x: str(x), win_df['Hs'].values))
        dat = [w for w in list(win_df['Win'].values)]
        sr = pd.Series(dat, idx, name='Win')
        win_sr = sr #.sort_index()
        self.win_sr = win_sr
        return win_sr
    
    def show1_series(self):
        if(self.win_df.empty): win_df = self.win_dataframe()
        else: win_df = self.win_df
        idx = list(map(lambda x: '(' + str(x) + ')', win_df['Hs'].values))
        show_sr = pd.Series(win_df['Show1'].values, idx, name='Show1')
        self.show1_sr = show_sr
        return show_sr
    
    def show2_series(self):
        if(self.win_df.empty): win_df = self.win_dataframe()
        else: win_df = self.win_df
        idx = list(map(lambda x: '(' + str(x) + ')', win_df['Hs'].values))
        show_sr = pd.Series(win_df['Show2'].values, idx, name='Show2')
        self.show2_sr = show_sr
        return show_sr
    
    def _dic_quinella(self, quinella_tags, headcount, qui_dic={}):
        if(len(quinella_tags)==0):
            return qui_dic
        Hc = headcount - len(quinella_tags)
        tags = quinella_tags.pop(0)
        tpl=[]
        for i, tag in enumerate(tags):
            if(tag.string==None):
                tpl.append((str(Hc) + "-" + str(i+(Hc+1)), float('0.0')))
            elif(not '.' in tag.string):
                tpl.append((str(Hc) + "-" + str(i+(Hc+1)), float('0.0')))
            else:
                tpl.append((str(Hc) + "-" + str(i+(Hc+1)), float(tag.string)))
        # tpl = [(str(Hc) + "-" + str(i+(Hc+1)), float(tag.string)) for i, tag in enumerate(tags)]
        qui_dic.update(tpl)
        return self._dic_quinella(quinella_tags, headcount)
    
    def _dic_exacta(self, exacta_tags, headcount, exa_dic={}):
        if(len(exacta_tags)==0):
            return exa_dic
        Hc = headcount - (len(exacta_tags) -1)
        lst = list(range(1, headcount+1))
        lst.remove(Hc)
        tags = exacta_tags.pop(0)
        tpl=[]
        for ls, tag in zip(lst, tags):
            if('.' in tag.string):
                tpl.append((str(Hc) + "->" + str(ls), float(tag.string)))
            else:
                tpl.append((str(Hc) + "->" + str(ls), float('0.0')))
        #tpl = [(str(Hc) + "->" + str(ls), float(tag.string)) for ls, tag in zip(lst, tags)]
        exa_dic.update(tpl)
        return self._dic_exacta(exacta_tags, headcount)  
    
    def _wide_dic(self, wide_tags, headcount, wide_dic={}):
        if(len(wide_tags)==0):
            return wide_dic
        Hc = headcount - len(wide_tags)
        tags = wide_tags.pop(0)
        tpl = []
        for i, tag in enumerate(tags):
            if('.' in tag.string):
                tpl.append((str(Hc) + "=" + str(i+(Hc+1)), float(tag.string)))
            else:
                tpl.append((str(Hc) + "=" + str(i+(Hc+1)), float('0.0')))
        # tpl = [(str(Hc) + "=" + str(i+(Hc+1)), float(tag.string)) for i,tag in enumerate(tags)]
        wide_dic.update(tpl)
        return self._wide_dic(wide_tags, headcount)
    
    def quinella_series(self):
        if(self._qui_soup==None): soup = self._set_quinella_soup()
        else: soup = self._qui_soup
        tables = soup.find_all("table", class_="oddsLs")
        quinella_tags = [tbl.find_all("td", class_=re.compile("txR")) for tbl in tables]
        headcount = len(quinella_tags) + 1
        qui_dic = self._dic_quinella(quinella_tags, headcount)
        qui_sr = pd.Series(qui_dic, name='Quinella')
        self.qui_sr = qui_sr
        return qui_sr
    
    def exacta_series(self):
        if(self._exa_soup==None): soup = self._set_exacta_soup()
        else: soup = self._exa_soup
        tables = soup.find_all("table", class_="oddsLs")
        exacta_tags = [tbl.find_all("td", class_=re.compile("txR")) for tbl in tables]
        headcount = len(exacta_tags)
        exa_dic = self._dic_exacta(exacta_tags, headcount)
        exa_sr = pd.Series(exa_dic, name='Exacta')
        self.exa_sr = exa_sr
        return exa_sr
        
    def wide_series(self): # 人気1位～18位
        if(self._wide_soup==None): soup = self._set_wide_soup()
        else: soup = self._wide_soup
        tables = soup.find_all("table", class_="oddsWLs")
        wide_tags = [tbl.find_all("td", class_=re.compile("txR"))[0::2] for tbl in tables]
        headcount = len(wide_tags) + 1
        wide_dic = self._wide_dic(wide_tags, headcount)
        wide_sr = pd.Series(wide_dic, name='Wide')
        self.wide_sr = wide_sr
        return wide_sr
    
    def win_fav(self):
        if(self.win_sr.empty): win_sr = self.win_series()
        else: win_sr = self.win_sr
        win_sr.sort_values()
        return win_sr
    
    def show1_fav(self):
        if(self.show1_sr.empty): show_sr = self.show1_series()
        else: show_sr = self.show1_sr
        show_sr.sort_values()
        return show_sr
    
    def show2_fav(self):
        if(self.show2_sr.empty): show_sr = self.show2_series()
        else: show_sr = self.show2_sr
        show_sr.sort_values()
        return show_sr
    
    def quinella_fav(self):
        if(self.qui_sr.empty): qui_sr = self.quinella_series()
        else: qui_sr = self.qui_sr
        qui_sort = qui_sr.sort_values()
        qui18_sr = qui_sort[0:18]
        return qui18_sr
    
    def exacta_fav(self):
        if(self.exa_sr.empty): exa_sr = self.exacta_series()
        else: exa_sr = self.exa_sr
        exa_sort = exa_sr.sort_values()
        exa18_sr = exa_sort[0:18]
        return exa18_sr
    
    def wide_fav(self):
        if(self.wide_sr.empty): wide_sr = self.wide_series()
        else: wide_sr = self.wide_sr
        wide_sort = wide_sr.sort_values()
        wide18_sr = wide_sort[0:18]
        return wide18_sr
    
    def _fav_dataframe(self, series): # 人気1位～18位
        if(self.horse_dic=={}): horse_dic = self._set_horse_dic()
        else: horse_dic = self.horse_dic
        if(self.win_sr.empty): win_sr = self.win_series()
        else: win_sr = self.win_sr
        sr = series
        sr_name= series.name
        sr_s = sr.sort_values()
        sr_18 = sr_s[0:18]
        idx18_sr = pd.Series(sr_18.index, range(1,19), name=sr_name)
        odds18_sr = pd.Series(sr_18.values, range(1,19), name='Odds')
        if(sr_name=='Quinella'):
            delimiter = '-'
        if(sr_name=='Exacta'):
            delimiter = '->'
        if(sr_name=='Wide'):
            delimiter = '='
        horse_pair = [(idx.split(delimiter)[0], idx.split(delimiter)[1])  for idx in idx18_sr]
        if(sr_name!='Exacta'):  # 人気順に入れ替え
            pair = []
            for tpl in horse_pair:
                if(win_sr[tpl[0]] < win_sr[tpl[1]]):
                    pair.append((tpl[0], tpl[1]))
                else:
                    pair.append((tpl[1], tpl[0]))
        else:
            pair = horse_pair
        horses = list(map(lambda x: f'{horse_dic[x[0]]} {delimiter} {horse_dic[x[1]]}', pair))
        horse_sr = pd.Series(horses, range(1,19), name='Horse')
        df = pd.DataFrame([horse_sr, idx18_sr, odds18_sr])
        fav18_df = df.T
        return fav18_df
    
    def quinella_dataframe(self):
        if(self.qui_sr.empty): quinella_sr = self.quinella_series()
        else: quinella_sr = self.qui_sr
        df = self._fav_dataframe(quinella_sr)
        return df
    
    def exacta_dataframe(self):
        if(self.exa_sr.empty): exacta_sr = self.exacta_series()
        else: exacta_sr = self.exa_sr
        df = self._fav_dataframe(exacta_sr)
        return df
    
    def wide_dataframe(self):
        if(self.wide_sr.empty): wide_sr = self.wide_series()
        else: wide_sr = self.wide_sr
        df = self._fav_dataframe(wide_sr)
        return df
    
    def _suitable_allocate(self, odds_list, money):
        """ 賭金の配分 返し例(3000, [1000, 700, 300], 6000) """
        roundup = lambda x: x - (x % 100) + 100 if((x % 100) != 0) else x # 100円切上げ
        odds_arr = np.array(list(map(lambda x: int(x * 10), odds_list))) # 整数型のarrayに変換
        lcm = np.lcm.reduce(odds_arr) # オッズの最小公倍数
        lcmPerOdds = [lcm / odds for odds in odds_arr] # 最小公倍数による逆数
        sum_lcmPerOdds = np.sum(lcmPerOdds) # 逆数の合計
        allocate_rate = [odds / sum_lcmPerOdds for odds in lcmPerOdds] # 合計が1になる割合
        allocate_money = [int(roundup(rate * money)) for rate in allocate_rate] # 資金の配分
        payout = [int(odds * money) for odds, money in zip(odds_list, allocate_money)] # 払い戻しのリスト
        modify_money = np.sum(allocate_money)
        return modify_money, allocate_money, np.min(payout)

    def simulation(self, bet_sr, money):
        modify_money, allocate_money, payout = self._suitable_allocate(list(bet_sr.values), money)
        """ dataframe """
        num_bet = len(bet_sr) + 1
        sr_index = range(1, num_bet)
        bet_lst = list(bet_sr.index)
        ticket_sr = pd.Series(bet_lst, sr_index, name='Ticket')
        allocate_sr = pd.Series(allocate_money, sr_index, name='Alloc')
        odds_sr = pd.Series(bet_sr.values, sr_index, name='Odds')
        payout_lst = [odds * alloc for (odds, alloc) in zip(list(odds_sr.values), allocate_money)]
        payout_sr = pd.Series(payout_lst, sr_index, name='Payout')
        head = []
        for bet in bet_lst:
            if('(' in bet):
                bet_head = re.sub('[()]', '', bet)
                head.append(int(''.join([bet_head, '00'])))
            elif('->' in bet):
                bet_head = bet.split('->')[0]
                head.append(int(''.join([bet_head, '000'])))
            elif('-' in bet and not '>' in bet):
                bet_head = bet.split('-')[0]
                head.append(int(''.join([bet_head, '0000'])))
            elif('=' in bet):
                bet_head = bet.split('=')[0]
                head.append(int(''.join([bet_head, '00000'])))
            else:
                head.append(int(bet))
        tail = []
        for bet in bet_lst:
            if('(' in bet):
                tail.append(int(0))
            elif('->' in bet):
                tail.append(int(bet.split('->')[1]))
            elif('-' in bet and not '>' in bet):
                tail.append(int(bet.split('-')[1]))
            elif('=' in bet):
                tail.append(int(bet.split('=')[1]))
            else:
                tail.append(int(0)) # 単勝
        head_sr = pd.Series(head, sr_index, name='head')
        tail_sr = pd.Series(tail, sr_index, name='tail')
        df = pd.DataFrame([head_sr, tail_sr, ticket_sr, allocate_sr, odds_sr, payout_sr])
        s_df = df.T.sort_values(['head', 'tail'])
        simulation_df = s_df.drop(columns=['head', 'tail'])
        profit_rate = payout / modify_money
        print(f"賭金 {modify_money:,}円 払戻 {payout:,}円 {len(s_df)} 点 戻率 {profit_rate:.1%}")
        display(simulation_df.style.set_properties(**{'text-align': 'right'}))


    def exacta_bettingall(self, exacta_sr, heads): # <- 買目のリストを渡す
        """ 総流しの馬単買目 """
        idx =[]
        head_lst = [str(Hs) + '-' for Hs in heads]
        for head in head_lst:
            boolst = exacta_sr.index.str.startswith(head)
            idx += [i for i, bol in enumerate(boolst) if(bol==True)]
            sr = exacta_sr.iloc[idx]
        return sr

    def exacta_bettingselect(self, exacta_sr, heads, tails):
        """ 頭と紐のリストで馬単買目 """
        def betting_list(heads, tails, exacta=[]):
            if(len(heads)==0):
                return exacta
            head = heads.pop(0)
            tail_ls = [t for t in tails if t != head]
            exacta += [ str(head) + '->' + str(tail) for tail in tail_ls]
            return betting_list(heads, tails)
        bet = betting_list(heads, tails)
        sr = exacta_sr[bet]
        return sr
    
    def qui_box(self, bet):
        lst = list(it.combinations(bet, 2))
        key = [str(l[0]) + '-' + str(l[1]) for l in lst] # 馬連
        qui = self.quinella_series()
        val = [qui[k] for k in key]
        sr = pd.Series(val, key)
        return sr
    
    def wide_box(self, bet):
        lst = list(it.combinations(bet, 2)) # ワイド
        key = [str(l[0]) + '=' + str(l[1]) for l in lst]
        qui = self.wide_series()
        val = [qui[k] for k in key]
        sr = pd.Series(val, key)
        return sr
    
    def tanpuku(self, bet):
        srw = self.win_series()
        srs = self.show2_series()
        dic = {}
        for b in bet:
            dic[str(b)] = srw[str(b)]
            dic['(' + str(b) + ')'] = srs['(' + str(b) + ')']
        sr = pd.Series(dic)
        return sr
    
    def qui_shed(self, bet, opponent):
        qsr = self.quinella_series()
        lst =[]
        for b in bet:
            for o in opp:
                if b < o:
                    lst.append(str(b) + '-' + str(o))
                else:
                    lst.append(str(o) + '-' + str(b))
        dic={}
        for l in lst:
            dic[l] = qsr[l]
        sr = pd.Series(dic)
        return sr

In [82]:
rc = mr.rc('東京sun') + '11'
sim = Betsimulation(rc)

In [139]:
sr2

1      2.8
7      4.8
5      5.1
4      7.5
2     11.2
9     14.8
12    19.4
3     19.5
10    21.1
11      45
6     45.9
13    73.3
8    152.4
Name: Win, dtype: float64

In [166]:
# sel = [1,7,8,9]
# list(it.permutations(bet, 3)) # 3連単
bet = [5, 9]
opp = [3, 4, 8]
def qui_shed(bet, opponent):
    qsr = sim.quinella_series()
    lst =[]
    for b in bet:
        for o in opp:
            if b < o:
                lst.append(str(b) + '-' + str(o))
            else:
                lst.append(str(o) + '-' + str(b))
    dic={}
    for l in lst:
        dic[l] = qsr[l]
    sr = pd.Series(dic)
    return sr
sim.simulation(sr, 5000)

賭金 5,300円 払戻 38,340円 6 点 戻率 723.4%


,Ticket,Alloc,Odds,Payout
1,3-5,700,59,41300
4,3-9,500,93.2,46600
2,4-5,2700,14.2,38340
5,4-9,1100,35.5,39050
3,5-8,200,331.2,66240
6,8-9,100,663.1,66310


In [59]:
def _suitable_allocate(self, odds_list, money):
    """ 賭金の配分 返し例(3000, [1000, 700, 300], 6000) """
    roundup = lambda x: x - (x % 100) + 100 if((x % 100) != 0) else x # 100円切上げ
    odds_arr = np.array(list(map(lambda x: int(x * 10), odds_list))) # 整数型のarrayに変換
    lcm = np.lcm.reduce(odds_arr) # オッズの最小公倍数
    lcmPerOdds = [lcm / odds for odds in odds_arr] # 最小公倍数による逆数
    sum_lcmPerOdds = np.sum(lcmPerOdds) # 逆数の合計
    allocate_rate = [odds / sum_lcmPerOdds for odds in lcmPerOdds] # 合計が1になる割合
    allocate_money = [int(roundup(rate * money)) for rate in allocate_rate] # 資金の配分
    payout = [int(odds * money) for odds, money in zip(odds_list, allocate_money)] # 払い戻しのリスト
    modify_money = np.sum(allocate_money)
    return modify_money, allocate_money, np.min(payout)

(1)     1.8
(7)     1.9
(5)     2.4
(4)     2.8
(2)     3.9
(9)     3.7
(12)    7.1
(3)     6.4
(10)      5
(11)   14.2
(6)    11.7
(13)   16.7
(8)    24.1
Name: Show2, dtype: float64

In [62]:
# tables = soup.find_all("table", class_="oddsLs")
# quinella_tags = [tbl.find_all("td", class_=re.compile("txR")) for tbl in tables]
# for tag in quinella_tags:
#     for t in tag:
#         if(t.string==None):
#             print(t.string)

None
None
None


In [26]:
# wide_sr = sim.wide_series()
# win_sr = sim.win_series()
# wn_sr = win_sr[0:3]
# wide_s = wide_sr.sort_values()
# exa_sr = sim.exacta_series()
# ex_sr = exa_sr[80:90]
# wd_sr = wide_s[20:28]
# qui_sr = sim.quinella_series()
# qu_sr = qui_sr[21:26]
# shw_sr = sim.show_series()
# sh_sr = shw_sr[0:6]
# sr = pd.concat([wn_sr, ex_sr, qu_sr, wd_sr, sh_sr])
# sim.simulation(sr, 100000)

In [ ]:
# """ 百万が表示できた！ """
# pd.options.display.float_format = '{:.7g}'.format # 百万の表示が可能

# def suitable_allocate(money, odds_list):
#     """ 賭金の配分 返し例(3000, [1000, 700, 300], 6000) """
#     roundup = lambda x: x - (x % 100) + 100 if((x % 100) != 0) else x # 100円切上げ
#     odds_arr = np.array(list(map(lambda x: int(x * 10), odds_list))) # 整数型のarrayに変換
#     lcm = np.lcm.reduce(odds_arr) # オッズの最小公倍数
#     lcmPerOdds = [lcm / odds for odds in odds_arr] # 最小公倍数による逆数
#     sum_lcmPerOdds = np.sum(lcmPerOdds) # 逆数の合計
#     allocate_rate = [odds / sum_lcmPerOdds for odds in lcmPerOdds] # 合計が1になる割合
#     allocate_money = [int(roundup(rate * money)) for rate in allocate_rate] # 資金の配分
#     payout = [int(odds * money) for odds, money in zip(odds_list, allocate_money)] # 払い戻しのリスト
#     modify_money = np.sum(allocate_money)
#     return modify_money, allocate_money, np.min(payout)

# def simulation(money, bet_sr):
#     modify_money, allocate_money, payout = suitable_allocate(money, list(bet_sr.values))
#     """ dataframe """
#     num_bet = len(bet_sr) + 1
#     sr_index = range(1, num_bet)
#     bet_lst = list(bet_sr.index)
#     ticket_sr = pd.Series(bet_lst, sr_index, name='Ticket')
#     allocate_sr = pd.Series(allocate_money, sr_index, name='Alloc')
#     odds_sr = pd.Series(bet_sr.values, sr_index, name='Odds')
#     payout_lst = [odds * alloc for (odds, alloc) in zip(list(odds_sr.values), allocate_money)]
#     payout_sr = pd.Series(payout_lst, sr_index, name='Payout')
#     deform_bet_lst = [''.join([bet,'0']) if('>' in bet) else bet for bet in bet_lst] # 馬連・馬単ソートのため
#     nonarrow_bet_lst = [bet.replace('>','') if('>' in bet) else bet for bet in deform_bet_lst]
#     head = list(map(lambda x: int(x.split('-')[0]) if('-' in x) else int(x), nonarrow_bet_lst))
#     head_sr = pd.Series(head, sr_index, name='head')
#     tail =  list(map(lambda x: int(x.split('-')[1]) if('-' in x) else int(0), nonarrow_bet_lst))
#     tail_sr = pd.Series(tail, sr_index, name='tail')
#     df = pd.DataFrame([head_sr, tail_sr, ticket_sr, allocate_sr, odds_sr, payout_sr])
#     simul_df = df.T.sort_values(['head', 'tail'])
#     simulation_df = simul_df.drop(columns=['head', 'tail'])
#     profit_rate = payout / modify_money
#     print(f"賭金 {modify_money:,}円 払戻 {payout:,}円 増分 {profit_rate:.1%}")
#     display(simulation_df)

#simulation(10000, sr[0:15])
#simulation(10000, c_sr)

# t_sr = pd.Series([3.3, 7.7], ['2-9', '2-11'])
# c_sr = pd.concat([sr[0:6], win_sr[0:7], t_sr])
# simulation(10000, exacta_sr[0:9])

#c_sr

In [ ]:
# def suitable_allocate(money, odds_list):
#     roundup = lambda x: x - (x % 100) + 100 if (x % 100) != 0 else x # 100円切上げ
#     odds_arr = np.array(list(map(lambda x: int(x * 10), odds_list))) # 整数型のarrayに変換
#     lcm = np.lcm.reduce(odds_arr) # オッズの最小公倍数(整数のみ)
#     lcmPerOdds = [lcm / odds for odds in odds_arr] # 最小公倍数による逆数
#     sum_lcmPerOdds = np.sum(lcmPerOdds) # 逆数の合計
#     allocate_rate = [odds / sum_lcmPerOdds for odds in lcmPerOdds] # 合計が1になる割合
#     allocate_money = [int(roundup(rate * money)) for rate in allocate_rate] # 資金の配分
#     payout = [int(odds * money) for odds, money in zip(odds_list, allocate_money)] # 払い戻しのリスト
#     modify_money = np.sum(allocate_money)
#     return modify_money, allocate_money, np.min(payout)
# odds = [1.6, 5.4, 7.4]
# money = 3000
# #list(map(lambda x: int(x * 10), odds))
# suitable_allocate(money, odds)